# Ridge regression

Ridge regression keeps the residual sum of squares and adds a penalty on the size of the slope. The penalty shrinks the slope toward zero and keeps the normal equations solvable when the columns are nearly dependent. The intercept should be left out of that penalty.


## The penalized slope

Center $x$ and $y$ first, so the intercept is zero in the centered coordinates and does not need a penalty. With one predictor, minimize

$$
\sum_{i=1}^{n}(d_i - b\, c_i)^2 + \lambda b^2,
$$

where $c_i = x_i - \bar x$ and $d_i = y_i - \bar y$. Differentiate with respect to $b$:

$$
-2\sum c_i(d_i - b c_i) + 2\lambda b = 0
\Longrightarrow
b(\,S_{xx} + \lambda\,) = S_{xy}
\Longrightarrow
b_{\lambda} = \dfrac{S_{xy}}{S_{xx} + \lambda}.
$$

The ordinary slope is the case $\lambda = 0$. For $\lambda > 0$, $|b_{\lambda}| < |b_0|$ whenever $S_{xy} \neq 0$. The sign is preserved and the magnitude shrinks.


## The three-point slope

The centered sums are $S_{xx} = 2$ and $S_{xy} = 3$, so the ordinary slope is $3/2$. At $\lambda = 1$,

$$
b_{1} = \dfrac{3}{2 + 1} = 1.
$$

At $\lambda = 4$,

$$
b_{4} = \dfrac{3}{2 + 4} = \dfrac{1}{2}.
$$

The predictions in centered coordinates are smaller than the ordinary-least-squares predictions. In the original coordinates the intercept remains $\bar y = 10/3$, because centering removed it from the penalty. The shrunk line is

$$
\widehat y = \dfrac{10}{3} + b_{\lambda}(x - 2).
$$


In [1]:
# Ridge on the centered slope: 3/(2+lambda). The intercept stays at the mean.
from fractions import Fraction
sxx, sxy = 2, 3
ordinary = Fraction(sxy, sxx)
ridge_1 = Fraction(sxy, sxx + 1)
ridge_4 = Fraction(sxy, sxx + 4)
print("ordinary", ordinary)
print("lambda 1", ridge_1)
print("lambda 4", ridge_4)
assert ordinary == Fraction(3, 2)
assert ridge_1 == 1
assert ridge_4 == Fraction(1, 2)
assert abs(ridge_4) < abs(ridge_1) < abs(ordinary)


ordinary 3/2
lambda 1 1
lambda 4 1/2


## Penalizing the intercept as well

If the penalty is applied to the uncentered coefficient vector, the equations become

$$
(X^{\mathsf T}X + \lambda I)\beta = X^{\mathsf T}y.
$$

For the three-point design and $\lambda = 1$,

$$
\begin{bmatrix} 4 & 6 \\ 6 & 15 \end{bmatrix}
\begin{bmatrix} b_0 \\ b_1 \end{bmatrix}
= \begin{bmatrix} 10 \\ 23 \end{bmatrix}.
$$

Elimination gives $b_1 = 4/3$ and $b_0 = 1/2$. The slope shrank from $3/2$ to $4/3$, which is a different value from the centered ridge slope $1$, and the intercept moved from $1/3$ to $1/2$. The penalty pulled the prediction at $x = 0$ toward zero even though the scientific target was the slope. Centering, then penalizing only the slope, avoids that pull.


In [2]:
# An uncentered penalty changes the intercept. The centered penalty does not.
from fractions import Fraction
import numpy as np
X = np.array([[1.0, 1], [1, 2], [1, 3]])
y = np.array([2.0, 3, 5])
beta = np.linalg.solve(X.T @ X + np.eye(2), X.T @ y)
print("penalized intercept and slope", beta)
assert np.allclose(beta, [1 / 2, 4 / 3])
assert not np.isclose(beta[0], 1 / 3)


penalized intercept and slope [0.5        1.33333333]


## Pitfall

Ridge at $\lambda = 0$ is ordinary least squares. Ridge as $\lambda \to \infty$ drives the centered slope to $0$ and predicts $\bar y$ for every $x$. Shrinkage is a continuous path between those two models, not a statement that the true slope is small. Choosing $\lambda$ because it produced a pleasant coefficient, without a held-out residual sum of squares or an equivalent criterion, is not an estimate of prediction error.


In [3]:
# Large lambda collapses the slope. Zero lambda returns ordinary least squares.
from fractions import Fraction
assert Fraction(3, 2 + 0) == Fraction(3, 2)
assert Fraction(3, 2 + 10_000) < Fraction(1, 1000)


## Summary

- After centering, $b_{\lambda} = S_{xy}/(S_{xx}+\lambda)$. On this table, $\lambda = 1$ gives slope $1$ and $\lambda = 4$ gives slope $1/2$.
- The centered intercept remains $\bar y$. Penalizing the raw intercept moves it; here it moved from $1/3$ to $1/2$.
- $\lambda = 0$ is least squares. A huge $\lambda$ predicts the mean and nothing else.
